# Example: BBBC014 — Dose-Response Morphological Profiling

Morphological profiling of distinct cell lines (A549 and MCF7) treated with various compounds across multiple concentrations to study dose-response effects.

**Dataset:** [BBBC014](https://bbbc.broadinstitute.org/BBBC014) (2-channel compound dose-response)

**Key Steps:**
1. **Metadata Prep:** Process annotations and calculate log concentrations for dose responses.
2. **Multi-Channel Processing:** Process full field-of-view 2-channel images.
3. **Dose-Response Analysis:** Extract embeddings and compute properties to differentiate cell line responses and visualize dose-dependent morphological shifts.

#0. Google Colab installation
Skip this step if the notebook is not running on Google Colab.

In [ ]:
# @title Install PhenoMe (Colab only)
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"PhenoMe requires Python 3.12+. Detected Python {sys.version_info.major}.{sys.version_info.minor}."
    )

if IN_COLAB:
    # Install dependencies without overriding Colab's pre-installed torch/notebook stack
    %pip install -q --upgrade-strategy only-if-needed "numpy>=1.24" pandas "matplotlib>=3.7" plotly scikit-learn joblib tqdm tifffile h5py scipy "scikit-image>=0.26.0" ipywidgets anywidget pillow "umap-learn==0.5.12" "nibabel>=5.2.0,<6" "dcor==0.7"
    %pip install -q --no-deps phenome
    try:
        import cv2  # noqa: F401
    except ImportError:
        %pip install -q opencv-python-headless
    print("✓ PhenoMe installed successfully! Proceed to Step 1.")
else:
    print("✓ Running in local environment. Proceed to Step 1.")

In [ ]:
# @title Download and unzip the image data into Colab's working space (2-5 minutes depending on the connection)

import gdown

## Link to BBBC014 dataset
url = "https://drive.google.com/file/d/1386ZTEw-dQ2CXJnNS6basC3dnz4tkqtj/view?usp=sharing"
output = "/content/dataset.zip"
gdown.download(url=url, output=output, quiet=False, fuzzy=True)

# Link to where the data will be stored
path2data = "/content/datasets/BBBC"
!mkdir -p $path2data

# Unzip data
!unzip -q $output -d $path2data

## Link to PhenoMe embeddings
url = "https://drive.google.com/file/d/1FadtWDhN3olvHrr5GXQG_8BIAiQiILJt/view?usp=sharing"
output = "/content/BBBC014_checkpoint_DINOv2_embeddings.h5"
gdown.download(url=url, output=output, quiet=False, fuzzy=True)

## Link to PhenoMe properties
url = "https://drive.google.com/file/d/1ONDExLd9QQ2ppfLPbrBBRp1m3aSxvNQ8/view?usp=sharing"
output = "/content/BBBC014_checkpoint_properties.h5"
gdown.download(url=url, output=output, quiet=False, fuzzy=True)

## 1. Paths & Configuration

Define the paths to the BBBC014 image dataset and the metadata CSV containing compound and concentration information.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

In [ ]:
IMAGE_DIR = "/content/datasets/BBBC/BBBC014/images"
METADATA_PATH = "/content/datasets/BBBC/BBBC014/metadata.csv"

## 2. Load Metadata & Preprocessing

We load the metadata and map the cell types to their respective line names (MCF7 and A549). We also calculate log concentrations to facilitate dose-response visualization.

In [ ]:
df_metadata = pd.read_csv(METADATA_PATH)

# Map cell types to line names
df_metadata["cell_line"] = df_metadata["cell_type"].apply(
    lambda x: "MCF7" if x in ["A", "B", "C", "D"] else "A549"
)

# Map concentration points to log concentrations
map_concentration = {
    1: -7.0, 2: -7.523, 3: -8.0, 4: -8.523, 5: -9.0, 6: -9.523,
    7: -10.0, 8: -10.523, 9: -11.0, 10: -11.523, 11: -12.0, 12: -13.0
}
df_metadata["log_concentration"] = df_metadata["concentration_point"].map(map_concentration)

df_metadata.head()

## 3. Multi-Channel Data Discovery

Since BBBC014 uses two channels per image, we specify both filename columns. PhenoMe will automatically load and merge these channels for processing.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    str(IMAGE_DIR),
    metadata_fn=make_dataframe_metadata_fn(
        df_metadata,
        filename_column=["filename_channel_1", "filename_channel_2"]
    ),
)
file_df.head()

## 4. Feature Extraction

We extract DINOv2 embeddings and compute a full set of morphological properties. Note the use of large `resize_size` and `pad_size` to accommodate full field-of-view images.

> **Tip — Checkpoints & Resuming**:
> - **Save / Resume during extraction**: Pass `checkpoint_path="embeddings.h5"` to `process_images` and `checkpoint_path="properties.h5"` to `compute_properties`. Embeddings and properties are written incrementally to HDF5, saving RAM and enabling automatic resume if interrupted.
> - **Load precomputed checkpoints**: If checkpoints were already computed, skip feature extraction entirely by loading them with `pheno.load_embeddings(...)` and `pheno.load_properties(...)` (or `pheno.load_results(...)`).

In [ ]:
path2embeddings = "/content/BBBC014_checkpoint_DINOv2_embeddings.h5"
path2properties = "/content/BBBC014_checkpoint_properties.h5"

# Option A: Load precomputed checkpoints (skip extraction)
# pheno.load_embeddings("/path/to/bbbc014/.../embeddings.h5")
# pheno.load_properties("/path/to/bbbc014/.../properties.h5")

# Option B: Extract features (optionally add checkpoint_path to save/resume)
pheno.process_images(
    wrapper,
    batch_size=8,
    channel_mode="split",
    resize_size=1358,
    pad_size=1360,
    l2_normalize_channels=False,
    checkpoint_path = path2embeddings # "bbbc014_embeddings.h5",  # Save embeddings incrementally to HDF5
)

df_properties = pheno.compute_properties(
    property_preset="complete",
    n_jobs=4,
    checkpoint_path = path2properties #"bbbc014_properties.h5",  # Save properties incrementally to HDF5
)

## 5. Visualization

Visualize the morphological differences between cell lines and observe dose-dependent shifts using t-SNE and UMAP.

In [ ]:
pheno.plot_tsne(
    color_by="cell_line",
    n_components=2,
)

In [ ]:
pheno.plot_umap(
    color_by="log_concentration",
    filters={"cell_line": "A549"},
    n_components=2,
)

## 6. Outlier Detection
Identify outliers based on embeddings to ensure robust analysis.

In [ ]:
outliers = pheno.detect_outliers(
    method="iqr",
    threshold=3,
    source="embeddings",
    group_by="cell_line",
    plot=True
)

## 7. Correlation Analysis
Compute correlations between morphological features and their embeddings to identify potential relationships.

In [ ]:
corr_results = pheno.compute_component_correlation(
    method="tsne",
    n_components=2,
    source="embeddings",
    filters={"cell_line": "A549"},
    top_k=5,
    correlation_method="pearson",
)

## 8. Interactive Exploration

Launch the interactive explorer to inspect the multi-channel images and their relative positions in the phenospace, highlighting how treatments affect cell morphology.

In [ ]:
# In Google Colab, use pheno.create_colab_interactive_explorer() instead.
e = pheno.create_interactive_explorer()